# sci_ai_engine — Phase A/B training on Colab

Designed to be re-run after a disconnect: every cell is idempotent, and
training always resumes from the last checkpoint saved to Google Drive.
**Runtime > Change runtime type > GPU** before running.

## 1. Mount Drive (checkpoints/data MUST live here, not local Colab disk)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/sci_ai_engine'
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/checkpoints/phaseA', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/checkpoints/phaseB', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data', exist_ok=True)
print('Drive ready at', DRIVE_ROOT)

## 2. Get the code (clone once; pulls latest on re-run)

In [ ]:
REPO_DIR = '/content/sci_ai_engine'
import os
if not os.path.exists(REPO_DIR):
    get_ipython().system(f'git clone https://github.com/sa9446/sci_ai.git {REPO_DIR}')
else:
    get_ipython().system(f'cd {REPO_DIR} && git pull')

## 3. Install training dependencies

In [ ]:
get_ipython().system(f'pip install -q -r {REPO_DIR}/model_training/requirements-train.txt')

## 4. One-time corpus prep (skip once train.bin/val.bin already exist on Drive)

The default fetch size (2000 arXiv abstracts/category, 300 Wikipedia
pages/category) turned out too small — on the AMD ROCm notebook it caused
Phase A to badly overfit by iteration ~500 (train loss collapsed to 0.05
while val loss never improved past its iter-250 value).
`--arxiv-max-per-category`/`--wikipedia-max-pages-per-category` below are
set much higher for a real run. If you already ran this cell with the old
small corpus, delete `{DRIVE_ROOT}/data/raw`, `{DRIVE_ROOT}/data/train.bin`,
`{DRIVE_ROOT}/data/val.bin`, `{DRIVE_ROOT}/tokenizer`, and
`{DRIVE_ROOT}/checkpoints/phaseA` first (the old checkpoint was trained on
the old vocab/corpus and isn't worth resuming from).

In [ ]:
DATA_DIR = f'{DRIVE_ROOT}/data'
if not os.path.exists(f'{DATA_DIR}/train.bin'):
    get_ipython().system(
        f'cd {REPO_DIR}/model_training && python data/fetch_corpus.py --out-dir {DATA_DIR}/raw '
        f'--arxiv-max-per-category 20000 --wikipedia-max-pages-per-category 3000'
    )
    get_ipython().system(f'cd {REPO_DIR}/model_training && python tokenizer/train_tokenizer.py --corpus-dir {DATA_DIR}/raw --out-dir {DRIVE_ROOT}/tokenizer')
    get_ipython().system(f'cd {REPO_DIR}/model_training && python data/prepare.py --raw-dir {DATA_DIR}/raw --tokenizer-path {DRIVE_ROOT}/tokenizer/tokenizer.json --out-dir {DATA_DIR}')
else:
    print('train.bin/val.bin already present on Drive — skipping corpus prep.')

## 5. Phase A pretraining (resumable — safe to re-run this cell after any disconnect)

In [ ]:
import json
with open(f'{DRIVE_ROOT}/tokenizer/tokenizer.json') as f:
    vocab_size = len(json.load(f)['model']['vocab'])
print('vocab_size =', vocab_size)

# batch-size lowered from 12->4 (grad-accum raised 8->24, same effective batch
# of 96) after a T4 (~14.56 GiB) OOM'd computing the loss at batch-size 12 —
# a (12, 1024, 32000) logits tensor alone is ~1.5 GiB, on top of ~13.5 GiB of
# activations across all 12 layers (no gradient checkpointing here). If this
# still OOMs, try --block-size 512 too (activation memory scales with it).
get_ipython().system(
    f'cd {REPO_DIR}/model_training && python train.py '
    f'--data-dir {DATA_DIR} --out-dir {DRIVE_ROOT}/checkpoints/phaseA --resume '
    f'--vocab-size {vocab_size} --max-iters 50000 --batch-size 4 --grad-accum-steps 24 '
    f'--eval-interval 250 --save-interval 250'
)

## 6. Synthetic Phase B dataset (no Anthropic dependency — hand-written physics formulas, not Claude-generated)

In [ ]:
get_ipython().system(
    f'cd {REPO_DIR}/model_training && python synth_dataset.py '
    f'--out-path {DRIVE_ROOT}/data/distilled.jsonl --num-queries 2000'
)

## 7. Phase B fine-tune (resumable, same pattern as Phase A)

In [ ]:
get_ipython().system(
    f'cd {REPO_DIR}/model_training && python finetune.py '
    f'--base-checkpoint {DRIVE_ROOT}/checkpoints/phaseA/ckpt_best.pt '
    f'--dataset-path {DRIVE_ROOT}/data/distilled.jsonl '
    f'--out-dir {DRIVE_ROOT}/checkpoints/phaseB --resume'
)